# Sensitivity Sampling — Colab
Multi-day noise + persistence sensitivity study. Checkpointed to Google Drive after every day.

## 1 — Mount Drive & install solver

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyomo', 'highspy'], check=True)
print('Dependencies ready.')

## 2 — Upload data file
Upload `combined_2025_with_frequency.csv` once — saved to Drive automatically.

In [ ]:
import os, shutil
from google.colab import files

DRIVE_DATA = '/content/drive/MyDrive/sensitivity_study/combined_2025_with_frequency.csv'
LOCAL_DATA = '/content/combined_2025_with_frequency.csv'
os.makedirs(os.path.dirname(DRIVE_DATA), exist_ok=True)

if os.path.exists(DRIVE_DATA):
    shutil.copy(DRIVE_DATA, LOCAL_DATA)
    print(f'Loaded from Drive: {DRIVE_DATA}')
else:
    print('File not found on Drive — upload it now.')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.copy(fname, LOCAL_DATA)
    shutil.copy(fname, DRIVE_DATA)
    print(f'Saved to Drive: {DRIVE_DATA}')

## 3 — Configuration

In [ ]:
import warnings, datetime, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint,
    NonNegativeReals, Reals, Binary, minimize, value, SolverFactory,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── Portfolio parameters ─────────────────────────────────────────────────
N_ECS = 10;  B_MAX_EC = 100.0;  S_MAX_EC = 200.0
ETA_MEAN = 0.95;  ETA_SIGMA = 0.0;  T_SUSTAIN = 0.5
P_MIN = 100.0;  SOC_INIT_FRAC = 0.5;  SOLVER = 'appsi_highs'
P_B_CHANCE = 0.6;  SCALE_MU = 1.0;  SCALE_SIGMA = 1.0
RANDOM_SEED = 42

P_BAR_AGG = N_ECS * B_MAX_EC
S_AGG     = N_ECS * S_MAX_EC

# ── Alpha values — SET DIRECTLY ──────────────────────────────────────────
ALPHAS = [0.0, 0.1, 0.25, 0.5, 1.0]

# ── Sampling ─────────────────────────────────────────────────────────────
N_SAMPLE_DAYS = 2       # ← change to 30 for full run
MULTI_SEED    = 999

# ── Output ───────────────────────────────────────────────────────────────
OUT_DIR = '/content/drive/MyDrive/sensitivity_study/results'
os.makedirs(OUT_DIR, exist_ok=True)
INDIVIDUAL_CSV = f'{OUT_DIR}/individual_day_results.csv'
AVERAGED_CSV   = f'{OUT_DIR}/averaged_results.csv'

ALL_VAR_KEYS = ['load', 'pv', 'spot',
                'fcrd_up_early', 'fcrd_dn_early',
                'fcrd_up_late',  'fcrd_dn_late']

VAR_LABELS = {
    'load': 'Load', 'pv': 'PV generation', 'spot': 'Spot price',
    'fcrd_up_early': 'FCR-D Up early', 'fcrd_dn_early': 'FCR-D Dn early',
    'fcrd_up_late':  'FCR-D Up late',  'fcrd_dn_late':  'FCR-D Dn late',
}

print(f'Portfolio: {P_BAR_AGG:.0f} kW / {S_AGG:.0f} kWh')
print(f'ALPHAS (set directly): {ALPHAS}')
print(f'Sampling {N_SAMPLE_DAYS} days  |  seed={MULTI_SEED}')
print(f'Results → {OUT_DIR}')

## 4 — Synthetic EC portfolio

In [ ]:
rng = np.random.default_rng(42)
EC_LIST = []
for i in range(N_ECS):
    base  = 'b' if rng.random() < P_B_CHANCE else 's'
    scale = -1.0
    while scale <= 0:
        scale = float(rng.normal(SCALE_MU, SCALE_SIGMA))
    EC_LIST.append({'id': f'ec{i:02d}', 'base': base, 'scale': scale,
                    'b_max': B_MAX_EC, 's_max': S_MAX_EC, 'eta': ETA_MEAN})
EC_IDS   = [ec['id'] for ec in EC_LIST]
EC_BY_ID = {ec['id']: ec for ec in EC_LIST}
n_b = sum(1 for e in EC_LIST if e['base'] == 'b')
print(f'{N_ECS} ECs: {n_b} b-type, {N_ECS-n_b} s-type')

## 5 — Load & preprocess data

In [ ]:
df_raw = pd.read_csv(LOCAL_DATA, parse_dates=['hour_utc'])
df_raw['spot_dkk']        = df_raw['spot_exkl_vat_ore_kwh'] / 100.0
df_raw['buy_price_dkk']   = df_raw['buy_price_inkl_vat_ore_kwh'] / 100.0
df_raw['sell_price_dkk']  = df_raw['sell_price_inkl_vat_ore_kwh'] / 100.0
df_raw['buy_markup_dkk']  = df_raw['buy_price_dkk']  - df_raw['spot_dkk']
df_raw['sell_markup_dkk'] = df_raw['sell_price_dkk'] - df_raw['spot_dkk']
df_raw['fcrd_up_early_dkk'] = df_raw['price_ore_kwh_fcr_d_upp__d_1_early'].fillna(0).clip(lower=0) / 100.0
df_raw['fcrd_up_late_dkk']  = df_raw['price_ore_kwh_fcr_d_upp__d_1_late' ].fillna(0).clip(lower=0) / 100.0
df_raw['fcrd_dn_early_dkk'] = df_raw['price_ore_kwh_fcr_d_ned__d_1_early'].fillna(0).clip(lower=0) / 100.0
df_raw['fcrd_dn_late_dkk']  = df_raw['price_ore_kwh_fcr_d_ned__d_1_late' ].fillna(0).clip(lower=0) / 100.0
df_raw['y_act_up'] = df_raw['y_act_fcrd_up'  ].fillna(0).clip(0, 1)
df_raw['y_act_dn'] = df_raw['y_act_fcrd_down'].fillna(0).clip(0, 1)
df = df_raw[df_raw['data_quality_flag'] == 0].copy()
df['date'] = df['hour_utc'].dt.date
df_b = df[df['ec_id'] == 'b'].set_index('hour_utc').sort_index()
df_s = df[df['ec_id'] == 's'].set_index('hour_utc').sort_index()
all_dates     = sorted(set(df_b.index.date))
all_dates_set = set(all_dates)
print(f'EC b: {len(df_b)} rows  |  EC s: {len(df_s)} rows')
print(f'Date range: {all_dates[0]} → {all_dates[-1]}')

## 6 — Empirical σ̂ (used by noise model, NOT to compute alphas)

In [ ]:
def compute_empirical_sigma(series):
    s = series.copy(); s.index = pd.to_datetime(s.index)
    diff = (s - s.shift(24)).dropna()
    return pd.DataFrame({'diff': diff.values, 'hour': diff.index.hour}).groupby('hour')['diff'].std().mean()

def compute_nrmse(forecast, true, mean_level):
    if mean_level == 0: return np.nan
    return np.sqrt(np.mean((forecast - true) ** 2)) / mean_level

def compute_nmae(forecast, true, mean_level):
    if mean_level == 0: return np.nan
    return np.mean(np.abs(forecast - true)) / mean_level

empirical_sigmas = {
    'load_b':        compute_empirical_sigma(df_b['consumption']),
    'load_s':        compute_empirical_sigma(df_s['consumption']),
    'pv_b':          compute_empirical_sigma(df_b['pv_production_kwh']),
    'pv_s':          compute_empirical_sigma(df_s['pv_production_kwh']),
    'spot':          compute_empirical_sigma(df_b['spot_dkk']),
    'fcrd_up_early': compute_empirical_sigma(df_b['fcrd_up_early_dkk']),
    'fcrd_dn_early': compute_empirical_sigma(df_b['fcrd_dn_early_dkk']),
    'fcrd_up_late':  compute_empirical_sigma(df_b['fcrd_up_late_dkk']),
    'fcrd_dn_late':  compute_empirical_sigma(df_b['fcrd_dn_late_dkk']),
}
mean_levels = {
    'load_b':        df_b['consumption'].mean(),
    'load_s':        df_s['consumption'].mean(),
    'pv_b':          df_b['pv_production_kwh'].mean(),
    'pv_s':          df_s['pv_production_kwh'].mean(),
    'spot':          df_b['spot_dkk'].mean(),
    'fcrd_up_early': df_b['fcrd_up_early_dkk'].mean(),
    'fcrd_dn_early': df_b['fcrd_dn_early_dkk'].mean(),
    'fcrd_up_late':  df_b['fcrd_up_late_dkk'].mean(),
    'fcrd_dn_late':  df_b['fcrd_dn_late_dkk'].mean(),
}
empirical_sigmas['pv'] = (empirical_sigmas['pv_b'] + empirical_sigmas['pv_s']) / 2.0
mean_levels['pv']      = (mean_levels['pv_b']      + mean_levels['pv_s'])      / 2.0

# NOTE: ALPHAS are NOT computed here — they are set directly in Cell 3
print('Empirical sigmas computed (used by noise model).')
print(f'ALPHAS in use (from Cell 3): {ALPHAS}')

## 7 — Noise & forecast functions

In [ ]:
def generate_random_walk_noise(horizon, sigma, rng):
    if sigma <= 0: return np.zeros(horizon)
    return np.cumsum(rng.normal(0, sigma, horizon))

def add_noise_to_vector(true_vec, horizon, sl_start, sl_end, alpha, sigma_hat, rng, scale=1.0):
    if not alpha or alpha <= 0: return true_vec.copy()
    noise = generate_random_walk_noise(horizon, alpha * sigma_hat, rng)
    return np.maximum(0, true_vec + noise[sl_start:sl_end] * scale)

def get_true_day(date):
    day_b = df_b[df_b.index.date == date].reset_index(drop=True)
    day_s = df_s[df_s.index.date == date].reset_index(drop=True)
    if len(day_b) != 24 or len(day_s) != 24: return None
    D_true, PV_true = {}, {}
    for ec in EC_LIST:
        src = day_b if ec['base'] == 'b' else day_s
        D_true[ec['id']]  = src['consumption'].values       * ec['scale']
        PV_true[ec['id']] = src['pv_production_kwh'].values * ec['scale']
    return {
        'D': D_true, 'PV': PV_true,
        'buy_price':    day_b['buy_price_dkk'].values,
        'sell_price':   day_b['sell_price_dkk'].values,
        'spot_dkk':     day_b['spot_dkk'].values,
        'buy_markup':   day_b['buy_markup_dkk'].values,
        'sell_markup':  day_b['sell_markup_dkk'].values,
        'fcrd_up_early':day_b['fcrd_up_early_dkk'].values,
        'fcrd_dn_early':day_b['fcrd_dn_early_dkk'].values,
        'fcrd_up_late': day_b['fcrd_up_late_dkk'].values,
        'fcrd_dn_late': day_b['fcrd_dn_late_dkk'].values,
        'y_act_up':     day_b['y_act_up'].values,
        'y_act_dn':     day_b['y_act_dn'].values,
        'date': date,
    }

def make_early_forecast(true_day, noise_config, rng):
    D_fc, PV_fc = {}, {}
    alpha_load = noise_config.get('load', 0)
    alpha_pv   = noise_config.get('pv', 0)
    for ec in EC_LIST:
        etype = ec['base']
        D_fc[ec['id']] = add_noise_to_vector(
            true_day['D'][ec['id']], 48, 24, 48,
            alpha_load, empirical_sigmas[f'load_{etype}'], rng, ec['scale'])
    if alpha_pv and alpha_pv > 0:
        sigma_pv   = alpha_pv * empirical_sigmas['pv']
        pv_noise48 = generate_random_walk_noise(48, sigma_pv, rng)
        pv_noise24 = pv_noise48[24:48]
        for ec in EC_LIST:
            PV_fc[ec['id']] = np.maximum(0, true_day['PV'][ec['id']] + pv_noise24 * ec['scale'])
    else:
        for ec in EC_LIST:
            PV_fc[ec['id']] = true_day['PV'][ec['id']].copy()
    alpha_spot = noise_config.get('spot', 0)
    if alpha_spot and alpha_spot > 0:
        spot_fc = add_noise_to_vector(true_day['spot_dkk'], 48, 24, 48,
                                      alpha_spot, empirical_sigmas['spot'], rng)
        buy_fc  = np.maximum(0, spot_fc + true_day['buy_markup'])
        sell_fc = np.maximum(0, spot_fc + true_day['sell_markup'])
    else:
        buy_fc  = true_day['buy_price'].copy()
        sell_fc = true_day['sell_price'].copy()
    alpha_fue        = noise_config.get('fcrd_up_early', 0)
    alpha_fde        = noise_config.get('fcrd_dn_early', 0)
    fcrd_up_early_fc = add_noise_to_vector(true_day['fcrd_up_early'], 48, 24, 48,
                                            alpha_fue, empirical_sigmas['fcrd_up_early'], rng)
    fcrd_dn_early_fc = add_noise_to_vector(true_day['fcrd_dn_early'], 48, 24, 48,
                                            alpha_fde, empirical_sigmas['fcrd_dn_early'], rng)
    return {
        'date': true_day['date'], 'D': D_fc, 'PV': PV_fc,
        'buy_price': buy_fc, 'sell_price': sell_fc,
        'fcrd_up_price': fcrd_up_early_fc, 'fcrd_dn_price': fcrd_dn_early_fc,
        'y_act_up': true_day['y_act_up'], 'y_act_dn': true_day['y_act_dn'],
    }

def make_late_forecast(true_day, noise_config, rng):
    D_fc, PV_fc = {}, {}
    alpha_load = noise_config.get('load', 0)
    alpha_pv   = noise_config.get('pv', 0)
    for ec in EC_LIST:
        etype = ec['base']
        D_fc[ec['id']] = add_noise_to_vector(
            true_day['D'][ec['id']], 30, 6, 30,
            alpha_load, empirical_sigmas[f'load_{etype}'], rng, ec['scale'])
    if alpha_pv and alpha_pv > 0:
        sigma_pv   = alpha_pv * empirical_sigmas['pv']
        pv_noise30 = generate_random_walk_noise(30, sigma_pv, rng)
        pv_noise24 = pv_noise30[6:30]
        for ec in EC_LIST:
            PV_fc[ec['id']] = np.maximum(0, true_day['PV'][ec['id']] + pv_noise24 * ec['scale'])
    else:
        for ec in EC_LIST:
            PV_fc[ec['id']] = true_day['PV'][ec['id']].copy()
    alpha_ful       = noise_config.get('fcrd_up_late', 0)
    alpha_fdl       = noise_config.get('fcrd_dn_late', 0)
    fcrd_up_late_fc = add_noise_to_vector(true_day['fcrd_up_late'], 30, 6, 30,
                                           alpha_ful, empirical_sigmas['fcrd_up_late'], rng)
    fcrd_dn_late_fc = add_noise_to_vector(true_day['fcrd_dn_late'], 30, 6, 30,
                                           alpha_fdl, empirical_sigmas['fcrd_dn_late'], rng)
    return {
        'date': true_day['date'], 'D': D_fc, 'PV': PV_fc,
        'buy_price':  true_day['buy_price'], 'sell_price': true_day['sell_price'],
        'fcrd_up_price_late':  fcrd_up_late_fc, 'fcrd_dn_price_late': fcrd_dn_late_fc,
        'fcrd_up_price_early': true_day['fcrd_up_early'],
        'fcrd_dn_price_early': true_day['fcrd_dn_early'],
        'y_act_up': true_day['y_act_up'], 'y_act_dn': true_day['y_act_dn'],
    }

print('Forecast functions defined.')

## 8 — Pyomo model builders

In [ ]:
def extract_early_accepted(m_early):
    acc_up, acc_dn = {}, {}
    for e in EC_IDS:
        for t in range(1, 25):
            acc_up[(e,t)] = max(0.0, value(m_early.p_res_up[e,t]))
            acc_dn[(e,t)] = max(0.0, value(m_early.p_res_dn[e,t]))
    return acc_up, acc_dn

def build_early(day, fcrd=True):
    m = ConcreteModel(); T = list(range(1, 25))
    m.E = Set(initialize=EC_IDS); m.T = Set(initialize=T)
    m.D      = Param(m.E, m.T, initialize={(e,t): day['D' ][e][t-1] for e in EC_IDS for t in T})
    m.PV     = Param(m.E, m.T, initialize={(e,t): day['PV'][e][t-1] for e in EC_IDS for t in T})
    m.b_max  = Param(m.E, initialize={e: EC_BY_ID[e]['b_max'] for e in EC_IDS})
    m.s_max  = Param(m.E, initialize={e: EC_BY_ID[e]['s_max'] for e in EC_IDS})
    m.eta    = Param(m.E, initialize={e: EC_BY_ID[e]['eta']   for e in EC_IDS})
    m.buy    = Param(m.T, initialize={t: day['buy_price' ][t-1] for t in T})
    m.sell   = Param(m.T, initialize={t: day['sell_price'][t-1] for t in T})
    if fcrd:
        m.lam_up   = Param(m.T, initialize={t: day['fcrd_up_price'][t-1] for t in T})
        m.lam_dn   = Param(m.T, initialize={t: day['fcrd_dn_price'][t-1] for t in T})
        m.y_act_up = Param(m.T, initialize={t: day['y_act_up'][t-1] for t in T})
        m.y_act_dn = Param(m.T, initialize={t: day['y_act_dn'][t-1] for t in T})
    m.p_im  = Var(m.E, m.T, domain=NonNegativeReals)
    m.p_ex  = Var(m.E, m.T, domain=NonNegativeReals)
    m.p_net = Var(m.E, m.T, domain=Reals)
    m.b_ch  = Var(m.E, m.T, domain=NonNegativeReals)
    m.b_dis = Var(m.E, m.T, domain=NonNegativeReals)
    m.soc   = Var(m.E, m.T, domain=NonNegativeReals)
    if fcrd:
        m.p_res_up = Var(m.E, m.T, domain=NonNegativeReals)
        m.p_res_dn = Var(m.E, m.T, domain=NonNegativeReals)
        m.p_act_up = Var(m.E, m.T, domain=NonNegativeReals)
        m.p_act_dn = Var(m.E, m.T, domain=NonNegativeReals)
        m.z_up = Var(m.T, domain=Binary); m.z_dn = Var(m.T, domain=Binary)
    if fcrd:
        m.obj = Objective(sense=minimize, expr=sum(
            m.buy[t]*(sum(m.p_im[e,t] for e in m.E) - sum(m.p_act_dn[e,t] for e in m.E))
          - m.sell[t]*(sum(m.p_ex[e,t] for e in m.E) - sum(m.p_act_up[e,t] for e in m.E))
          - m.lam_up[t]*sum(m.p_res_up[e,t] for e in m.E)
          - m.lam_dn[t]*sum(m.p_res_dn[e,t] for e in m.E)
          for t in m.T))
    else:
        m.obj = Objective(sense=minimize, expr=sum(
            m.buy[t]*m.p_im[e,t] - m.sell[t]*m.p_ex[e,t] for e in m.E for t in m.T))
    m.c_m1b = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_net[e,t]-m.p_im[e,t]+m.p_ex[e,t]==0)
    def _c2i(m,e,t):
        rhs = m.p_net[e,t]+m.PV[e,t]-m.D[e,t]-m.b_ch[e,t]+m.b_dis[e,t]
        if fcrd: rhs = rhs - m.p_act_dn[e,t] + m.p_act_up[e,t]
        return rhs == 0
    m.c_c2i = Constraint(m.E, m.T, rule=_c2i)
    if fcrd:
        m.c_c3b_up = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_act_up[e,t]==m.y_act_up[t]*m.p_res_up[e,t])
        m.c_c3b_dn = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_act_dn[e,t]==m.y_act_dn[t]*m.p_res_dn[e,t])
    def _m2j(m,e,t):
        if t==1: return Constraint.Skip
        gain  = m.b_ch[e,t]  + (m.p_act_dn[e,t] if fcrd else 0)
        drain = m.b_dis[e,t] + (m.p_act_up[e,t] if fcrd else 0)
        return m.soc[e,t] == m.soc[e,t-1] + m.eta[e]*gain - (1/m.eta[e])*drain
    m.c_m2j = Constraint(m.E, m.T, rule=_m2j)
    def _m2k(m,e):
        gain  = m.b_ch[e,1]  + (m.p_act_dn[e,1] if fcrd else 0)
        drain = m.b_dis[e,1] + (m.p_act_up[e,1] if fcrd else 0)
        return m.soc[e,1] == SOC_INIT_FRAC*m.s_max[e] + m.eta[e]*gain - (1/m.eta[e])*drain
    m.c_m2k   = Constraint(m.E, rule=_m2k)
    m.c_b_ch  = Constraint(m.E, m.T, rule=lambda m,e,t: m.b_ch[e,t]  <= m.b_max[e])
    m.c_b_dis = Constraint(m.E, m.T, rule=lambda m,e,t: m.b_dis[e,t] <= m.b_max[e])
    m.c_soc   = Constraint(m.E, m.T, rule=lambda m,e,t: m.soc[e,t]   <= m.s_max[e])
    if fcrd:
        m.c_res_up = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_res_up[e,t] <= m.b_max[e])
        m.c_res_dn = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_res_dn[e,t] <= m.b_max[e])
        M_big = P_BAR_AGG
        m.c_min_up = Constraint(m.T, rule=lambda m,t: sum(m.p_res_up[e,t] for e in m.E) >= P_MIN*m.z_up[t])
        m.c_min_dn = Constraint(m.T, rule=lambda m,t: sum(m.p_res_dn[e,t] for e in m.E) >= P_MIN*m.z_dn[t])
        m.c_max_up = Constraint(m.T, rule=lambda m,t: sum(m.p_res_up[e,t] for e in m.E) <= M_big*m.z_up[t])
        m.c_max_dn = Constraint(m.T, rule=lambda m,t: sum(m.p_res_dn[e,t] for e in m.E) <= M_big*m.z_dn[t])
    return m

def build_late(day_late, early_acc_up, early_acc_dn, early_clearing_up, early_clearing_dn):
    m = ConcreteModel(); T = list(range(1, 25))
    m.E = Set(initialize=EC_IDS); m.T = Set(initialize=T)
    m.D      = Param(m.E, m.T, initialize={(e,t): day_late['D' ][e][t-1] for e in EC_IDS for t in T})
    m.PV     = Param(m.E, m.T, initialize={(e,t): day_late['PV'][e][t-1] for e in EC_IDS for t in T})
    m.b_max  = Param(m.E, initialize={e: EC_BY_ID[e]['b_max'] for e in EC_IDS})
    m.s_max  = Param(m.E, initialize={e: EC_BY_ID[e]['s_max'] for e in EC_IDS})
    m.eta    = Param(m.E, initialize={e: EC_BY_ID[e]['eta']   for e in EC_IDS})
    m.buy    = Param(m.T, initialize={t: day_late['buy_price' ][t-1] for t in T})
    m.sell   = Param(m.T, initialize={t: day_late['sell_price'][t-1] for t in T})
    m.lam_up_late  = Param(m.T, initialize={t: day_late['fcrd_up_price_late'][t-1] for t in T})
    m.lam_dn_late  = Param(m.T, initialize={t: day_late['fcrd_dn_price_late'][t-1] for t in T})
    m.lam_up_early = Param(m.T, initialize={t: early_clearing_up[t-1] for t in T})
    m.lam_dn_early = Param(m.T, initialize={t: early_clearing_dn[t-1] for t in T})
    m.lam_up_bb = Param(m.T, initialize={t: max(early_clearing_up[t-1], day_late['fcrd_up_price_late'][t-1]) for t in T})
    m.lam_dn_bb = Param(m.T, initialize={t: max(early_clearing_dn[t-1], day_late['fcrd_dn_price_late'][t-1]) for t in T})
    m.p_acc_up = Param(m.E, m.T, initialize=early_acc_up)
    m.p_acc_dn = Param(m.E, m.T, initialize=early_acc_dn)
    m.y_act_up = Param(m.T, initialize={t: day_late['y_act_up'][t-1] for t in T})
    m.y_act_dn = Param(m.T, initialize={t: day_late['y_act_dn'][t-1] for t in T})
    m.p_im  = Var(m.E, m.T, domain=NonNegativeReals); m.p_ex  = Var(m.E, m.T, domain=NonNegativeReals)
    m.p_net = Var(m.E, m.T, domain=Reals)
    m.b_ch  = Var(m.E, m.T, domain=NonNegativeReals); m.b_dis = Var(m.E, m.T, domain=NonNegativeReals)
    m.soc   = Var(m.E, m.T, domain=NonNegativeReals)
    m.c_up  = Var(m.E, m.T, domain=NonNegativeReals); m.c_dn  = Var(m.E, m.T, domain=NonNegativeReals)
    m.a_up  = Var(m.E, m.T, domain=NonNegativeReals); m.a_dn  = Var(m.E, m.T, domain=NonNegativeReals)
    m.p_res_up = Var(m.E, m.T, domain=NonNegativeReals); m.p_res_dn = Var(m.E, m.T, domain=NonNegativeReals)
    m.p_act_up = Var(m.E, m.T, domain=NonNegativeReals); m.p_act_dn = Var(m.E, m.T, domain=NonNegativeReals)
    m.z_up = Var(m.T, domain=Binary); m.z_dn = Var(m.T, domain=Binary)
    m.w_up = Var(m.T, domain=Binary); m.w_dn = Var(m.T, domain=Binary)
    m.c_l1_up = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_res_up[e,t]==m.p_acc_up[e,t]-m.c_up[e,t]+m.a_up[e,t])
    m.c_l1_dn = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_res_dn[e,t]==m.p_acc_dn[e,t]-m.c_dn[e,t]+m.a_dn[e,t])
    m.c_l2_up = Constraint(m.E, m.T, rule=lambda m,e,t: m.c_up[e,t] <= m.p_acc_up[e,t])
    m.c_l2_dn = Constraint(m.E, m.T, rule=lambda m,e,t: m.c_dn[e,t] <= m.p_acc_dn[e,t])
    m.c_no_mix_up = Constraint(m.E, m.T, rule=lambda m,e,t: m.c_up[e,t] <= m.p_acc_up[e,t]*(1-m.w_up[t]))
    m.c_no_mix_dn = Constraint(m.E, m.T, rule=lambda m,e,t: m.c_dn[e,t] <= m.p_acc_dn[e,t]*(1-m.w_dn[t]))
    m.obj = Objective(sense=minimize, expr=sum(
          m.buy[t]*(sum(m.p_im[e,t] for e in m.E) - sum(m.p_act_dn[e,t] for e in m.E))
        - m.sell[t]*(sum(m.p_ex[e,t] for e in m.E) - sum(m.p_act_up[e,t] for e in m.E))
        - sum(m.lam_up_early[t]*m.p_acc_up[e,t] for e in m.E)
        - sum(m.lam_dn_early[t]*m.p_acc_dn[e,t] for e in m.E)
        + sum(m.lam_up_bb[t]*m.c_up[e,t] - m.lam_up_late[t]*m.a_up[e,t] for e in m.E)
        + sum(m.lam_dn_bb[t]*m.c_dn[e,t] - m.lam_dn_late[t]*m.a_dn[e,t] for e in m.E)
        for t in m.T))
    m.c_m1b = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_net[e,t]-m.p_im[e,t]+m.p_ex[e,t]==0)
    def _c2i(m,e,t):
        return m.p_net[e,t]+m.PV[e,t]-m.D[e,t]-m.b_ch[e,t]+m.b_dis[e,t]-m.p_act_dn[e,t]+m.p_act_up[e,t]==0
    m.c_c2i   = Constraint(m.E, m.T, rule=_c2i)
    m.c_c3_up = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_act_up[e,t]==m.y_act_up[t]*m.p_res_up[e,t])
    m.c_c3_dn = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_act_dn[e,t]==m.y_act_dn[t]*m.p_res_dn[e,t])
    def _m2j(m,e,t):
        if t==1: return Constraint.Skip
        return m.soc[e,t]==m.soc[e,t-1]+m.eta[e]*(m.b_ch[e,t]+m.p_act_dn[e,t])-(1/m.eta[e])*(m.b_dis[e,t]+m.p_act_up[e,t])
    m.c_m2j = Constraint(m.E, m.T, rule=_m2j)
    def _m2k(m,e):
        return m.soc[e,1]==SOC_INIT_FRAC*m.s_max[e]+m.eta[e]*(m.b_ch[e,1]+m.p_act_dn[e,1])-(1/m.eta[e])*(m.b_dis[e,1]+m.p_act_up[e,1])
    m.c_m2k   = Constraint(m.E, rule=_m2k)
    m.c_b_ch  = Constraint(m.E, m.T, rule=lambda m,e,t: m.b_ch[e,t]  <= m.b_max[e])
    m.c_b_dis = Constraint(m.E, m.T, rule=lambda m,e,t: m.b_dis[e,t] <= m.b_max[e])
    m.c_soc   = Constraint(m.E, m.T, rule=lambda m,e,t: m.soc[e,t]   <= m.s_max[e])
    m.c_res_up = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_res_up[e,t] <= m.b_max[e])
    m.c_res_dn = Constraint(m.E, m.T, rule=lambda m,e,t: m.p_res_dn[e,t] <= m.b_max[e])
    M_big = P_BAR_AGG
    m.c_min_up = Constraint(m.T, rule=lambda m,t: sum(m.p_res_up[e,t] for e in m.E) >= P_MIN*m.z_up[t])
    m.c_min_dn = Constraint(m.T, rule=lambda m,t: sum(m.p_res_dn[e,t] for e in m.E) >= P_MIN*m.z_dn[t])
    m.c_max_up = Constraint(m.T, rule=lambda m,t: sum(m.p_res_up[e,t] for e in m.E) <= M_big*m.z_up[t])
    m.c_max_dn = Constraint(m.T, rule=lambda m,t: sum(m.p_res_dn[e,t] for e in m.E) <= M_big*m.z_dn[t])
    m.c_top_min_up = Constraint(m.T, rule=lambda m,t: sum(m.a_up[e,t] for e in m.E) >= P_MIN*m.w_up[t])
    m.c_top_min_dn = Constraint(m.T, rule=lambda m,t: sum(m.a_dn[e,t] for e in m.E) >= P_MIN*m.w_dn[t])
    m.c_top_max_up = Constraint(m.T, rule=lambda m,t: sum(m.a_up[e,t] for e in m.E) <= M_big*m.w_up[t])
    m.c_top_max_dn = Constraint(m.T, rule=lambda m,t: sum(m.a_dn[e,t] for e in m.E) <= M_big*m.w_dn[t])
    return m

print('Model builders defined.')

## 9 — Simulation, pipeline & persistence

In [ ]:
def solve_model(m):
    from pyomo.opt import TerminationCondition
    result = SolverFactory(SOLVER).solve(m, tee=False)
    tc = result.solver.termination_condition
    if tc not in (TerminationCondition.optimal,
                  TerminationCondition.locallyOptimal,
                  TerminationCondition.feasible):
        return None
    return result

def simulate_day(m_late, true_day):
    true_buy = true_day['buy_price'];  true_sell = true_day['sell_price']
    true_fcrd_up_early = true_day['fcrd_up_early']; true_fcrd_dn_early = true_day['fcrd_dn_early']
    true_fcrd_up_late  = true_day['fcrd_up_late'];  true_fcrd_dn_late  = true_day['fcrd_dn_late']
    true_y_act_up = true_day['y_act_up']; true_y_act_dn = true_day['y_act_dn']
    no_bat_cost = 0.0
    for h in range(24):
        D_agg  = sum(true_day['D'][e][h]  for e in EC_IDS)
        PV_agg = sum(true_day['PV'][e][h] for e in EC_IDS)
        no_bat_cost += true_buy[h]*max(0,D_agg-PV_agg) - true_sell[h]*max(0,PV_agg-D_agg)
    actual_arb_cost=0.0; actual_fcrd_up_net=0.0; actual_fcrd_dn_net=0.0
    for t in range(1,25):
        h=t-1
        res_up=sum(value(m_late.p_res_up[e,t]) for e in EC_IDS)
        res_dn=sum(value(m_late.p_res_dn[e,t]) for e in EC_IDS)
        acc_up=sum(value(m_late.p_acc_up[e,t]) for e in EC_IDS)
        acc_dn=sum(value(m_late.p_acc_dn[e,t]) for e in EC_IDS)
        c_up=sum(value(m_late.c_up[e,t]) for e in EC_IDS)
        c_dn=sum(value(m_late.c_dn[e,t]) for e in EC_IDS)
        a_up=sum(value(m_late.a_up[e,t]) for e in EC_IDS)
        a_dn=sum(value(m_late.a_dn[e,t]) for e in EC_IDS)
        act_up=true_y_act_up[h]*res_up; act_dn=true_y_act_dn[h]*res_dn
        actual_p_im=0.0; actual_p_ex=0.0
        for e in EC_IDS:
            b_ch_e=value(m_late.b_ch[e,t]); b_dis_e=value(m_late.b_dis[e,t])
            act_up_e=true_y_act_up[h]*value(m_late.p_res_up[e,t])
            act_dn_e=true_y_act_dn[h]*value(m_late.p_res_dn[e,t])
            p_net_e=true_day['D'][e][h]-true_day['PV'][e][h]+b_ch_e+act_dn_e-b_dis_e-act_up_e
            actual_p_im+=max(0,p_net_e); actual_p_ex+=max(0,-p_net_e)
        actual_arb_cost+=true_buy[h]*(actual_p_im-act_dn)-true_sell[h]*(actual_p_ex-act_up)
        rev_early_up=true_fcrd_up_early[h]*acc_up; rev_early_dn=true_fcrd_dn_early[h]*acc_dn
        bb_up=max(true_fcrd_up_early[h],true_fcrd_up_late[h])
        bb_dn=max(true_fcrd_dn_early[h],true_fcrd_dn_late[h])
        actual_fcrd_up_net+=rev_early_up-bb_up*c_up+true_fcrd_up_late[h]*a_up
        actual_fcrd_dn_net+=rev_early_dn-bb_dn*c_dn+true_fcrd_dn_late[h]*a_dn
    total_value=(no_bat_cost-actual_arb_cost)+actual_fcrd_up_net+actual_fcrd_dn_net
    return {'no_bat_cost':no_bat_cost,'arb_cost':actual_arb_cost,
            'arb_saving':no_bat_cost-actual_arb_cost,
            'fcrd_up_net':actual_fcrd_up_net,'fcrd_dn_net':actual_fcrd_dn_net,
            'total_value':total_value}

def _nrmse_block(early_fc, late_fc, true_day):
    D_true_agg  =np.array([sum(true_day['D'][e][h]  for e in EC_IDS) for h in range(24)])
    PV_true_agg =np.array([sum(true_day['PV'][e][h] for e in EC_IDS) for h in range(24)])
    D_early_agg =np.array([sum(early_fc['D'][e][h]  for e in EC_IDS) for h in range(24)])
    PV_early_agg=np.array([sum(early_fc['PV'][e][h] for e in EC_IDS) for h in range(24)])
    D_late_agg  =np.array([sum(late_fc['D'][e][h]   for e in EC_IDS) for h in range(24)])
    PV_late_agg =np.array([sum(late_fc['PV'][e][h]  for e in EC_IDS) for h in range(24)])
    return {
        'load_early':    compute_nrmse(D_early_agg,  D_true_agg,  max(D_true_agg.mean(),  1e-6)),
        'load_late':     compute_nrmse(D_late_agg,   D_true_agg,  max(D_true_agg.mean(),  1e-6)),
        'pv_early':      compute_nrmse(PV_early_agg, PV_true_agg, max(PV_true_agg.mean(), 1e-6)),
        'pv_late':       compute_nrmse(PV_late_agg,  PV_true_agg, max(PV_true_agg.mean(), 1e-6)),
        'spot':          compute_nrmse(early_fc['buy_price'],       true_day['buy_price'],    max(true_day['buy_price'].mean(),    1e-6)),
        'fcrd_up_early': compute_nrmse(early_fc['fcrd_up_price'],   true_day['fcrd_up_early'],max(true_day['fcrd_up_early'].mean(),1e-6)),
        'fcrd_dn_early': compute_nrmse(early_fc['fcrd_dn_price'],   true_day['fcrd_dn_early'],max(true_day['fcrd_dn_early'].mean(),1e-6)),
        'fcrd_up_late':  compute_nrmse(late_fc['fcrd_up_price_late'],true_day['fcrd_up_late'], max(true_day['fcrd_up_late'].mean(), 1e-6)),
        'fcrd_dn_late':  compute_nrmse(late_fc['fcrd_dn_price_late'],true_day['fcrd_dn_late'], max(true_day['fcrd_dn_late'].mean(), 1e-6)),
    }

def run_pipeline(date, noise_config, seed):
    true_day = get_true_day(date)
    if true_day is None: return None
    rng = np.random.default_rng(seed)
    early_fc = make_early_forecast(true_day, noise_config, rng)
    m_early  = build_early(early_fc, fcrd=True)
    if solve_model(m_early) is None: return None
    acc_up, acc_dn = extract_early_accepted(m_early)
    late_fc = make_late_forecast(true_day, noise_config, rng)
    m_late  = build_late(late_fc, acc_up, acc_dn, true_day['fcrd_up_early'], true_day['fcrd_dn_early'])
    if solve_model(m_late) is None: return None
    result = simulate_day(m_late, true_day)
    result['date']  = date
    result['nrmse'] = _nrmse_block(early_fc, late_fc, true_day)
    return result

def get_reference_day(date, lag_days):
    from datetime import timedelta
    ref_date = date - timedelta(days=lag_days)
    return get_true_day(ref_date) if ref_date in all_dates_set else None

def make_persistence_early(true_day, ref_day):
    return {
        'date': true_day['date'],
        'D':  {ec['id']: ref_day['D'][ec['id']].copy()  for ec in EC_LIST},
        'PV': {ec['id']: ref_day['PV'][ec['id']].copy() for ec in EC_LIST},
        'buy_price':     ref_day['buy_price'].copy(),
        'sell_price':    ref_day['sell_price'].copy(),
        'fcrd_up_price': ref_day['fcrd_up_early'].copy(),
        'fcrd_dn_price': ref_day['fcrd_dn_early'].copy(),
        'y_act_up': true_day['y_act_up'], 'y_act_dn': true_day['y_act_dn'],
    }

def make_persistence_late(true_day, ref_day):
    return {
        'date': true_day['date'],
        'D':  {ec['id']: ref_day['D'][ec['id']].copy()  for ec in EC_LIST},
        'PV': {ec['id']: ref_day['PV'][ec['id']].copy() for ec in EC_LIST},
        'buy_price':  true_day['buy_price'], 'sell_price': true_day['sell_price'],
        'fcrd_up_price_late':  ref_day['fcrd_up_late'].copy(),
        'fcrd_dn_price_late':  ref_day['fcrd_dn_late'].copy(),
        'fcrd_up_price_early': true_day['fcrd_up_early'],
        'fcrd_dn_price_early': true_day['fcrd_dn_early'],
        'y_act_up': true_day['y_act_up'], 'y_act_dn': true_day['y_act_dn'],
    }

def run_persistence_pipeline(date, lag_days):
    true_day = get_true_day(date); ref_day = get_reference_day(date, lag_days)
    if true_day is None or ref_day is None: return None
    early_fc = make_persistence_early(true_day, ref_day)
    m_early  = build_early(early_fc, fcrd=True)
    if solve_model(m_early) is None: return None
    acc_up, acc_dn = extract_early_accepted(m_early)
    late_fc = make_persistence_late(true_day, ref_day)
    m_late  = build_late(late_fc, acc_up, acc_dn, true_day['fcrd_up_early'], true_day['fcrd_dn_early'])
    if solve_model(m_late) is None: return None
    result = simulate_day(m_late, true_day)
    result['date'] = date; result['lag_days'] = lag_days
    result['nrmse'] = _nrmse_block(early_fc, late_fc, true_day)
    return result

print('All pipeline functions defined.')

## 10 — Run sampling loop
~3–4 min per day. **Checkpointed to Drive after every day.**

In [ ]:
# ── Sample candidate dates ────────────────────────────────────────────────
candidate_dates = [
    d for d in all_dates
    if (d - datetime.timedelta(days=2)) in all_dates_set
    and (d - datetime.timedelta(days=7)) in all_dates_set
]
rng_sample    = np.random.default_rng(MULTI_SEED)
n             = min(N_SAMPLE_DAYS, len(candidate_dates))
sampled_dates = sorted(rng_sample.choice(len(candidate_dates), size=n, replace=False))
sampled_dates = [candidate_dates[i] for i in sampled_dates]
print(f'Sampled {len(sampled_dates)} days from {candidate_dates[0]} to {candidate_dates[-1]}')
print('Dates:', sampled_dates)
print(f'ALPHAS: {ALPHAS}')

# ── Storage (matches original notebook exactly) ──────────────────────────
daily_data_all  = {a: [] for a in ALPHAS}
daily_data_oaat = {var: {a: [] for a in ALPHAS} for var in ALL_VAR_KEYS}
daily_rev_daily, daily_rev_weekly     = [], []
daily_nrmse_daily, daily_nrmse_weekly = [], []

# ── Resume from checkpoint ────────────────────────────────────────────────
completed_dates = set()
if os.path.exists(INDIVIDUAL_CSV):
    _df_prev = pd.read_csv(INDIVIDUAL_CSV)
    completed_dates = set(pd.to_datetime(_df_prev['date']).dt.date)
    print(f'Resuming: {len(completed_dates)} days already done.')

individual_rows = []
t_start = time.time()

for i_day, date in enumerate(sampled_dates):
    if date in completed_dates:
        print(f'  [{i_day+1}/{len(sampled_dates)}] {date} — skipped (already done)')
        continue

    t0 = time.time()
    print(f'  [{i_day+1}/{len(sampled_dates)}] {date} ...', end=' ', flush=True)

    # Perfect-foresight baseline
    bl = run_pipeline(date, {k: 0.0 for k in ALL_VAR_KEYS}, RANDOM_SEED)
    if bl is None:
        print('skip (no data / solver failed)')
        continue
    bl_rev = bl['total_value']

    # Persistence revenues
    rd = run_persistence_pipeline(date, lag_days=2)
    rw = run_persistence_pipeline(date, lag_days=7)
    if rd is not None and bl_rev:
        daily_rev_daily.append(rd['total_value'] / bl_rev)
        daily_nrmse_daily.append(np.mean(list(rd['nrmse'].values())))
    if rw is not None and bl_rev:
        daily_rev_weekly.append(rw['total_value'] / bl_rev)
        daily_nrmse_weekly.append(np.mean(list(rw['nrmse'].values())))

    # All-at-once noise sweep
    for alpha in ALPHAS:
        nc = {k: alpha for k in ALL_VAR_KEYS}
        r  = run_pipeline(date, nc, RANDOM_SEED)
        if r is not None and bl_rev:
            r['alpha']    = alpha
            r['rev_norm'] = r['total_value'] / bl_rev
            daily_data_all[alpha].append(r)
            individual_rows.append({
                'date': str(date), 'sweep': 'all', 'var': 'all', 'alpha': alpha,
                'rev_norm': r['rev_norm'], 'total_value': r['total_value'],
                'arb_saving': r['arb_saving'], 'fcrd_up_net': r['fcrd_up_net'],
                'fcrd_dn_net': r['fcrd_dn_net'],
                **{f'nrmse_{k}': v for k,v in r['nrmse'].items()},
            })

    # One-at-a-time noise sweep
    for var in ALL_VAR_KEYS:
        for alpha in ALPHAS:
            nc = {v: 0.0 for v in ALL_VAR_KEYS}; nc[var] = alpha
            r  = run_pipeline(date, nc, RANDOM_SEED)
            if r is not None and bl_rev:
                r['alpha']    = alpha
                r['rev_norm'] = r['total_value'] / bl_rev
                daily_data_oaat[var][alpha].append(r)
                individual_rows.append({
                    'date': str(date), 'sweep': 'oaat', 'var': var, 'alpha': alpha,
                    'rev_norm': r['rev_norm'], 'total_value': r['total_value'],
                    'arb_saving': r['arb_saving'], 'fcrd_up_net': r['fcrd_up_net'],
                    'fcrd_dn_net': r['fcrd_dn_net'],
                    **{f'nrmse_{k}': v for k,v in r['nrmse'].items()},
                })

    # Persistence rows
    for lag, rd_rw in [('D-2', rd), ('D-7', rw)]:
        if rd_rw is not None and bl_rev:
            individual_rows.append({
                'date': str(date), 'sweep': 'persistence', 'var': lag,
                'alpha': np.nan,
                'rev_norm': rd_rw['total_value'] / bl_rev,
                'total_value': rd_rw['total_value'],
                'arb_saving': rd_rw['arb_saving'],
                'fcrd_up_net': rd_rw['fcrd_up_net'],
                'fcrd_dn_net': rd_rw['fcrd_dn_net'],
                **{f'nrmse_{k}': v for k,v in rd_rw['nrmse'].items()},
            })

    elapsed = time.time() - t0
    pd.DataFrame(individual_rows).to_csv(INDIVIDUAL_CSV, index=False)
    print(f'done ({elapsed:.0f}s) — checkpoint saved')

print(f'\nAll done in {(time.time()-t_start)/60:.1f} min.')

## 11 — Average across sampled days

In [ ]:
avg_rev_daily   = np.nanmean(daily_rev_daily)   if daily_rev_daily   else np.nan
avg_rev_weekly  = np.nanmean(daily_rev_weekly)  if daily_rev_weekly  else np.nan
avg_nrmse_daily  = np.nanmean(daily_nrmse_daily)  if daily_nrmse_daily  else np.nan
avg_nrmse_weekly = np.nanmean(daily_nrmse_weekly) if daily_nrmse_weekly else np.nan

# All-at-once averages
avg_results_all = []
for alpha in ALPHAS:
    records = daily_data_all[alpha]
    if not records: continue
    avg_results_all.append({
        'alpha':    alpha,
        'rev_norm': np.nanmean([r['rev_norm'] for r in records]),
        'nrmse':    {k: np.nanmean([r['nrmse'][k] for r in records if k in r['nrmse']])
                     for k in records[0]['nrmse']},
    })
df_avg_all = pd.DataFrame(avg_results_all)

# One-at-a-time averages
avg_results_oaat = {}
df_avg_oaat = {}
for var in ALL_VAR_KEYS:
    rows = []
    for alpha in ALPHAS:
        records = daily_data_oaat[var][alpha]
        if not records: continue
        nrmse_key = 'load_early' if var == 'load' else ('pv_early' if var == 'pv' else var)
        rows.append({
            'alpha':    alpha,
            'rev_norm': np.nanmean([r['rev_norm'] for r in records]),
            'nrmse':    np.nanmean([r['nrmse'].get(nrmse_key, np.nan) for r in records]),
        })
    avg_results_oaat[var] = rows
    df_avg_oaat[var] = pd.DataFrame(rows)

print(f'Averaged results over {len(sampled_dates)} sampled days:')
print(f'  Daily persistence (D-2):  rev_norm={avg_rev_daily:.3f},  avg nRMSE={avg_nrmse_daily:.4f}')
print(f'  Weekly persistence (D-7): rev_norm={avg_rev_weekly:.3f}, avg nRMSE={avg_nrmse_weekly:.4f}')
print()
print('All-at-once:')
print(df_avg_all[['alpha','rev_norm']].to_string(index=False))

# Save averaged CSV
avg_rows_csv = []
for r in avg_results_all:
    avg_rows_csv.append({'sweep':'all','var':'all','alpha':r['alpha'],'rev_norm':r['rev_norm']})
for var in ALL_VAR_KEYS:
    for r in avg_results_oaat[var]:
        avg_rows_csv.append({'sweep':'oaat','var':var,'alpha':r['alpha'],'rev_norm':r['rev_norm'],'nrmse':r['nrmse']})
avg_rows_csv.append({'sweep':'persistence','var':'D-2','rev_norm':avg_rev_daily,'nrmse':avg_nrmse_daily})
avg_rows_csv.append({'sweep':'persistence','var':'D-7','rev_norm':avg_rev_weekly,'nrmse':avg_nrmse_weekly})
pd.DataFrame(avg_rows_csv).to_csv(AVERAGED_CSV, index=False)
print(f'\nSaved: {AVERAGED_CSV}')

## 12 — Summary plot (Revenue vs α  &  Revenue vs nRMSE)

In [ ]:
C_PV            = '#FFCC00'
C_FCRD_UP_EARLY = '#FF7F0E'
C_FCRD_UP_LATE  = '#D62728'
C_LOAD          = '#1F77B4'
C_FCRD_DN_EARLY = '#00BDF2'
C_FCRD_DN_LATE  = '#2CA02C'
C_SPOT          = '#7F7F7F'
C_DAILY         = '#F42CFF'
C_WEEKLY        = '#9E00A3'

COLORS = {
    'load':          C_LOAD,
    'pv':            C_PV,
    'spot':          C_SPOT,
    'fcrd_up_early': C_FCRD_UP_EARLY,
    'fcrd_dn_early': C_FCRD_DN_EARLY,
    'fcrd_up_late':  C_FCRD_UP_LATE,
    'fcrd_dn_late':  C_FCRD_DN_LATE,
}

fig, axes = plt.subplots(1, 2, figsize=(21, 7))
fig.patch.set_facecolor('white')
fig.suptitle(
    f'Revenue Sensitivity & Forecast Quality — {len(sampled_dates)}-day average '
    f'(seed={MULTI_SEED})',
    fontsize=14, fontweight='bold', y=0.98
)

# ── Plot 1: Revenue vs α ────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('white')

for var in ALL_VAR_KEYS:
    d = df_avg_oaat[var]
    ax.plot(d['alpha'], d['rev_norm'], marker='o', lw=2, ms=5,
            color=COLORS[var], label=VAR_LABELS[var])
ax.plot(df_avg_all['alpha'], df_avg_all['rev_norm'],
        marker='s', lw=3, ms=8, color='#333333', label='All simultaneous', zorder=10)

ax.axhline(avg_rev_daily,  color=C_DAILY,  ls='--', lw=2, alpha=0.85,
           label=f'Daily persist. D-2 ({avg_rev_daily:.1%})')
ax.axhline(avg_rev_weekly, color=C_WEEKLY, ls='--', lw=2, alpha=0.85,
           label=f'Weekly persist. D-7 ({avg_rev_weekly:.1%})')
ax.axhline(1.0, color='gray', ls=':', lw=1)

ax.set_xlabel('α (fraction of σ̂)', fontsize=11)
ax.set_ylabel('Revenue (fraction of perfect)', fontsize=11)
ax.set_title('Revenue vs α  [multi-day avg]', fontsize=12, fontweight='bold')
ax.legend(fontsize=7, ncol=2)
ax.grid(False)
ax.spines[['top', 'right']].set_visible(False)

# ── Plot 2: Revenue vs nRMSE ────────────────────────────────────────────
ax = axes[1]
ax.set_facecolor('white')

for var in ALL_VAR_KEYS:
    d = df_avg_oaat[var]
    ax.plot(d['nrmse'], d['rev_norm'], marker='o', lw=2, ms=5,
            color=COLORS[var], label=VAR_LABELS[var])

avg_nrmses_all = [np.mean(list(r['nrmse'].values())) for r in avg_results_all]
ax.plot(avg_nrmses_all, df_avg_all['rev_norm'],
        marker='s', lw=3, ms=8, color='#333333', label='All simultaneous', zorder=10)

ax.scatter([avg_nrmse_daily],  [avg_rev_daily],  color=C_DAILY,  s=200, marker='*',
           zorder=15, edgecolors='black', lw=0.8, label='Daily persist.')
ax.scatter([avg_nrmse_weekly], [avg_rev_weekly], color=C_WEEKLY, s=200, marker='*',
           zorder=15, edgecolors='black', lw=0.8, label='Weekly persist.')
ax.annotate(f'D-2\n({avg_rev_daily:.1%})', (avg_nrmse_daily, avg_rev_daily),
            textcoords='offset points', xytext=(8, -12), fontsize=8, color=C_DAILY)
ax.annotate(f'D-7\n({avg_rev_weekly:.1%})', (avg_nrmse_weekly, avg_rev_weekly),
            textcoords='offset points', xytext=(8, -12), fontsize=8, color=C_WEEKLY)

ax.axhline(1.0, color='gray', ls=':', lw=1)
ax.set_xlabel('nRMSE (forecast quality)', fontsize=11)
ax.set_ylabel('Revenue (fraction of perfect)', fontsize=11)
ax.set_title('Revenue vs nRMSE  [multi-day avg]', fontsize=12, fontweight='bold')
ax.legend(fontsize=7, ncol=2)
ax.grid(False)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
PLOT_PATH = f'{OUT_DIR}/sensitivity_summary.png'
plt.savefig(PLOT_PATH, dpi=150)
plt.show()
print(f'Plot saved: {PLOT_PATH}')